In [1]:
!pip install -q kaggle pandas openpyxl

In [2]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [3]:
import os
import shutil

# Create Kaggle configuration folder
os.makedirs("/root/.kaggle", exist_ok=True)

# Move kaggle.json
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")

# Set permission
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API configured successfully!")

Kaggle API configured successfully!


In [4]:
!kaggle datasets list -s "online retail"

ref                                                             title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  ------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
tunguz/online-retail                                            Online Retail                                        7471504  2021-04-12 21:49:08.737000          16540        130                1  
ulrikthygepedersen/online-retail-dataset                        Online Retail Dataset                                7742495  2023-01-20 13:50:59.410000          28476        180                1  
mashlyn/online-retail-ii-uci                                    Online Retail II UCI                                15217139  2019-12-02 11:03:36.200000          53426        283                1  
ishanshriv

In [6]:
!kaggle datasets download -d vijayuv/onlineretail

Dataset URL: https://www.kaggle.com/datasets/vijayuv/onlineretail
License(s): CC0-1.0
100% 7.20M/7.20M [00:00<00:00, 81.8MB/s]



In [7]:
import zipfile
import os

with zipfile.ZipFile("onlineretail.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

print(os.listdir("dataset"))

['OnlineRetail.csv']


In [8]:
import pandas as pd
import numpy as np
import json
import sqlite3
import os

In [9]:
print(os.listdir("dataset"))

['OnlineRetail.csv']


In [15]:
df = pd.read_csv(
    "dataset/OnlineRetail.csv",
    encoding="latin1"
)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [16]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
df.info()

Dataset Shape: (541909, 8)

Column Names:
Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [17]:
print(df.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [18]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


In [19]:
transactions = df[
    [
        "InvoiceNo",
        "StockCode",
        "CustomerID",
        "Quantity",
        "InvoiceDate"
    ]
]

transactions.to_csv(
    "transactions.csv",
    index=False
)

print("transactions.csv created successfully!")

transactions.head()

transactions.csv created successfully!


,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
0,536365,85123A,17850.0,6,12/1/2010 8:26
1,536365,71053,17850.0,6,12/1/2010 8:26
2,536365,84406B,17850.0,8,12/1/2010 8:26
3,536365,84029G,17850.0,6,12/1/2010 8:26
4,536365,84029E,17850.0,6,12/1/2010 8:26


In [20]:
products = df[
    [
        "StockCode",
        "Description",
        "UnitPrice"
    ]
].copy()

In [21]:
products = products.drop_duplicates(
    subset=["StockCode"]
)

In [22]:
products.to_json(
    "products.json",
    orient="records",
    indent=4
)

print("products.json created successfully!")

products.head()

products.json created successfully!


,StockCode,Description,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,71053,WHITE METAL LANTERN,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39


In [23]:
customers = df[
    [
        "CustomerID",
        "Country"
    ]
].copy()

In [24]:
customers = customers.drop_duplicates(
    subset=["CustomerID"]
)

In [25]:
customers.to_excel(
    "customers.xlsx",
    index=False
)

print("customers.xlsx created successfully!")

customers.head()

customers.xlsx created successfully!


,CustomerID,Country
0,17850.0,United Kingdom
9,13047.0,United Kingdom
26,12583.0,France
46,13748.0,United Kingdom
65,15100.0,United Kingdom


In [26]:
transactions = pd.read_csv(
    "transactions.csv"
)

products = pd.read_json(
    "products.json"
)

customers = pd.read_excel(
    "customers.xlsx"
)

In [27]:
print("TRANSACTIONS")
print(transactions.shape)
display(transactions.head())

print("\nPRODUCTS")
print(products.shape)
display(products.head())

print("\nCUSTOMERS")
print(customers.shape)
display(customers.head())

TRANSACTIONS
(541909, 5)


,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
0,536365,85123A,17850.0,6,12/1/2010 8:26
1,536365,71053,17850.0,6,12/1/2010 8:26
2,536365,84406B,17850.0,8,12/1/2010 8:26
3,536365,84029G,17850.0,6,12/1/2010 8:26
4,536365,84029E,17850.0,6,12/1/2010 8:26



PRODUCTS
(4070, 3)


,StockCode,Description,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,71053,WHITE METAL LANTERN,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39



CUSTOMERS
(4373, 2)


,CustomerID,Country
0,17850.0,United Kingdom
1,13047.0,United Kingdom
2,12583.0,France
3,13748.0,United Kingdom
4,15100.0,United Kingdom


In [28]:
print("TRANSACTIONS")
print(transactions.shape)
display(transactions.head())

print("\nPRODUCTS")
print(products.shape)
display(products.head())

print("\nCUSTOMERS")
print(customers.shape)
display(customers.head())

TRANSACTIONS
(541909, 5)


,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
0,536365,85123A,17850.0,6,12/1/2010 8:26
1,536365,71053,17850.0,6,12/1/2010 8:26
2,536365,84406B,17850.0,8,12/1/2010 8:26
3,536365,84029G,17850.0,6,12/1/2010 8:26
4,536365,84029E,17850.0,6,12/1/2010 8:26



PRODUCTS
(4070, 3)


,StockCode,Description,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,71053,WHITE METAL LANTERN,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39



CUSTOMERS
(4373, 2)


,CustomerID,Country
0,17850.0,United Kingdom
1,13047.0,United Kingdom
2,12583.0,France
3,13748.0,United Kingdom
4,15100.0,United Kingdom


In [29]:
print(
    "Duplicate Transactions:",
    transactions.duplicated().sum()
)

print(
    "Duplicate Products:",
    products.duplicated().sum()
)

print(
    "Duplicate Customers:",
    customers.duplicated().sum()
)

Duplicate Transactions: 5429
Duplicate Products: 0
Duplicate Customers: 0


In [30]:
transactions = transactions.dropna(
    subset=[
        "InvoiceNo",
        "StockCode",
        "CustomerID",
        "Quantity",
        "InvoiceDate"
    ]
)

In [31]:
transactions = transactions.dropna(
    subset=[
        "InvoiceNo",
        "StockCode",
        "CustomerID",
        "Quantity",
        "InvoiceDate"
    ]
)

In [32]:
transactions = transactions[
    transactions["Quantity"] > 0
]

In [33]:
print("Transactions Shape:", transactions.shape)

transactions.head()

Transactions Shape: (397924, 5)


,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
0,536365,85123A,17850.0,6,12/1/2010 8:26
1,536365,71053,17850.0,6,12/1/2010 8:26
2,536365,84406B,17850.0,8,12/1/2010 8:26
3,536365,84029G,17850.0,6,12/1/2010 8:26
4,536365,84029E,17850.0,6,12/1/2010 8:26


In [34]:
print(products.isnull().sum())

StockCode        0
Description    176
UnitPrice        0
dtype: int64


In [36]:
products = products.dropna(
    subset=[
        "StockCode",
        "Description",
        "UnitPrice"
    ]
)

In [37]:
products = products.drop_duplicates(
    subset=["StockCode"]
)

In [38]:
products = products[
    products["UnitPrice"] > 0
]

In [39]:
print("Products Shape:", products.shape)

products.head()

Products Shape: (3855, 3)


,StockCode,Description,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,71053,WHITE METAL LANTERN,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39


In [40]:
print("Products Shape:", products.shape)

products.head()

Products Shape: (3855, 3)


,StockCode,Description,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,71053,WHITE METAL LANTERN,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39


In [41]:
customers = customers.drop_duplicates(
    subset=["CustomerID"]
)

In [42]:
print("Customers Shape:", customers.shape)

customers.head()

Customers Shape: (4373, 2)


,CustomerID,Country
0,17850.0,United Kingdom
1,13047.0,United Kingdom
2,12583.0,France
3,13748.0,United Kingdom
4,15100.0,United Kingdom


In [43]:
merged_data = pd.merge(
    transactions,
    products,
    on="StockCode",
    how="left"
)

merged_data.head()

,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice
0,536365,85123A,17850.0,6,12/1/2010 8:26,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,536365,71053,17850.0,6,12/1/2010 8:26,WHITE METAL LANTERN,3.39
2,536365,84406B,17850.0,8,12/1/2010 8:26,CREAM CUPID HEARTS COAT HANGER,2.75
3,536365,84029G,17850.0,6,12/1/2010 8:26,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,536365,84029E,17850.0,6,12/1/2010 8:26,RED WOOLLY HOTTIE WHITE HEART.,3.39


In [44]:
final_data = pd.merge(
    merged_data,
    customers,
    on="CustomerID",
    how="left"
)

final_data.head()

,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country
0,536365,85123A,17850.0,6,12/1/2010 8:26,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom
1,536365,71053,17850.0,6,12/1/2010 8:26,WHITE METAL LANTERN,3.39,United Kingdom
2,536365,84406B,17850.0,8,12/1/2010 8:26,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom
3,536365,84029G,17850.0,6,12/1/2010 8:26,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom
4,536365,84029E,17850.0,6,12/1/2010 8:26,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom


In [45]:
print(
    "Final Dataset Shape:",
    final_data.shape
)

print("\nNumber of Rows:",
      final_data.shape[0])

print("Number of Columns:",
      final_data.shape[1])

Final Dataset Shape: (397924, 8)

Number of Rows: 397924
Number of Columns: 8


In [46]:
unmatched_products = final_data[
    final_data["Description"].isnull()
]

print(
    "Unmatched Product Records:",
    len(unmatched_products)
)

Unmatched Product Records: 4877


In [47]:
unmatched_customers = final_data[
    final_data["Country"].isnull()
]

print(
    "Unmatched Customer Records:",
    len(unmatched_customers)
)

Unmatched Customer Records: 0


In [48]:
final_data = final_data.dropna(
    subset=[
        "Description",
        "UnitPrice",
        "Country"
    ]
)

print(
    "Final Cleaned Dataset Shape:",
    final_data.shape
)

Final Cleaned Dataset Shape: (393047, 8)


In [49]:
final_data["Revenue"] = (
    final_data["Quantity"]
    *
    final_data["UnitPrice"]
)

In [50]:
final_data.head()

,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
0,536365,85123A,17850.0,6,12/1/2010 8:26,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
1,536365,71053,17850.0,6,12/1/2010 8:26,WHITE METAL LANTERN,3.39,United Kingdom,20.34
2,536365,84406B,17850.0,8,12/1/2010 8:26,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
3,536365,84029G,17850.0,6,12/1/2010 8:26,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
4,536365,84029E,17850.0,6,12/1/2010 8:26,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34


In [51]:
print(final_data.info())

display(final_data.head())

<class 'pandas.core.frame.DataFrame'>
Index: 393047 entries, 0 to 397923
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    393047 non-null  object 
 1   StockCode    393047 non-null  object 
 2   CustomerID   393047 non-null  float64
 3   Quantity     393047 non-null  int64  
 4   InvoiceDate  393047 non-null  object 
 5   Description  393047 non-null  object 
 6   UnitPrice    393047 non-null  float64
 7   Country      393047 non-null  object 
 8   Revenue      393047 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 30.0+ MB
None


,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
0,536365,85123A,17850.0,6,12/1/2010 8:26,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
1,536365,71053,17850.0,6,12/1/2010 8:26,WHITE METAL LANTERN,3.39,United Kingdom,20.34
2,536365,84406B,17850.0,8,12/1/2010 8:26,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
3,536365,84029G,17850.0,6,12/1/2010 8:26,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
4,536365,84029E,17850.0,6,12/1/2010 8:26,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34


In [52]:
total_revenue = final_data[
    "Revenue"
].sum()

print(
    "Total Sales Revenue:",
    round(total_revenue, 2)
)

Total Sales Revenue: 10781828.79


In [53]:
top_products = (
    final_data
    .groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

top_products

,Description,Revenue
0,"PAPER CRAFT , LITTLE BIRDIE",168469.60
1,PARTY BUNTING,142549.40
2,REGENCY CAKESTAND 3 TIER,135911.40
3,WHITE HANGING HEART T-LIGHT HOLDER,93794.10
4,MEDIUM CERAMIC TOP STORAGE JAR,81032.64


In [54]:
top_countries = (
    final_data
    .groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

top_countries

,Country,Revenue
0,United Kingdom,8889920.304
1,Netherlands,363884.480
2,EIRE,331956.410
3,Germany,264070.690
4,France,227053.310


In [55]:
top_customers = (
    final_data
    .groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

top_customers

,CustomerID,Revenue
0,18102.0,408759.96
1,14646.0,357531.09
2,17450.0,186197.03
3,14911.0,182797.24
4,16446.0,168472.50


In [56]:
customer_revenue = (
    final_data
    .groupby("CustomerID")["Revenue"]
    .sum()
    .reset_index()
)

customer_revenue.head()

,CustomerID,Revenue
0,12346.0,77183.60
1,12347.0,5438.44
2,12348.0,1790.16
3,12349.0,1926.96
4,12350.0,408.04


In [57]:
conditions = [
    customer_revenue["Revenue"] < 1000,

    (
        customer_revenue["Revenue"] >= 1000
    )
    &
    (
        customer_revenue["Revenue"] < 5000
    ),

    (
        customer_revenue["Revenue"] >= 5000
    )
    &
    (
        customer_revenue["Revenue"] < 20000
    ),

    customer_revenue["Revenue"] >= 20000
]

categories = [
    "Low Value",
    "Medium Value",
    "High Value",
    "Premium"
]

customer_revenue[
    "Customer_Category"
] = np.select(
    conditions,
    categories,
    default="Unknown"
)

customer_revenue.head(10)

,CustomerID,Revenue,Customer_Category
0,12346.0,77183.60,Premium
1,12347.0,5438.44,High Value
2,12348.0,1790.16,Medium Value
3,12349.0,1926.96,Medium Value
4,12350.0,408.04,Low Value
5,12352.0,1619.07,Medium Value
6,12353.0,171.04,Low Value
7,12354.0,1241.44,Medium Value
8,12355.0,545.74,Low Value
9,12356.0,3708.92,Medium Value


In [58]:
customer_revenue[
    "Customer_Category"
].value_counts()

,count
Customer_Category,
Low Value,2443
Medium Value,1546
High Value,297
Premium,53


In [59]:
customer_revenue[
    "Customer_Category"
].value_counts()

,count
Customer_Category,
Low Value,2443
Medium Value,1546
High Value,297
Premium,53


In [62]:
country_revenue = (
    final_data
    .groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

print(country_revenue)

Country
United Kingdom          8889920.304
Netherlands              363884.480
EIRE                     331956.410
Germany                  264070.690
France                   227053.310
Australia                173998.110
Spain                     67443.040
Switzerland               66619.970
Japan                     48600.220
Belgium                   47858.020
Sweden                    43673.140
Norway                    40305.600
Portugal                  32720.610
Finland                   23306.790
Channel Islands           22069.620
Denmark                   21291.790
Italy                     21065.450
Cyprus                    16917.610
Singapore                 11502.980
Poland                    10264.540
Austria                    9883.010
Israel                     9095.940
Greece                     6128.800
Iceland                    5438.440
Canada                     4315.900
USA                        4138.830
Unspecified                2961.750
Malta               

In [63]:
high_market = country_revenue.idxmax()

high_market_revenue = country_revenue.max()

print("High Performing Market:", high_market)
print("Revenue:", round(high_market_revenue, 2))

High Performing Market: United Kingdom
Revenue: 8889920.3


In [64]:
underperforming_market = country_revenue.idxmin()

underperforming_revenue = country_revenue.min()

print("Underperforming Market:", underperforming_market)
print("Revenue:", round(underperforming_revenue, 2))

Underperforming Market: Saudi Arabia
Revenue: 181.44


In [65]:
# 1. Calculate revenue by country
country_revenue = (
    final_data
    .groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

# 2. High-performing market
high_market = country_revenue.idxmax()
high_market_revenue = country_revenue.max()

print("High Performing Market:", high_market)
print("Revenue:", round(high_market_revenue, 2))

# 3. Underperforming market
underperforming_market = country_revenue.idxmin()
underperforming_revenue = country_revenue.min()

print("\nUnderperforming Market:", underperforming_market)
print("Revenue:", round(underperforming_revenue, 2))

High Performing Market: United Kingdom
Revenue: 8889920.3

Underperforming Market: Saudi Arabia
Revenue: 181.44


In [66]:
final_data["Revenue"] = (
    final_data["Quantity"] * final_data["UnitPrice"]
)

In [67]:
final_data["Revenue"] = (
    final_data["Quantity"] * final_data["UnitPrice"]
)

In [69]:
conn = sqlite3.connect("retail_sales.db")

final_data.to_sql(
    "retail_sales",
    conn,
    if_exists="replace",
    index=False
)

print("Final data stored successfully in retail_sales table!")

Final data stored successfully in retail_sales table!


In [70]:
query = """
SELECT *
FROM retail_sales
LIMIT 5;
"""

sql_data = pd.read_sql_query(query, conn)

display(sql_data)

,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
0,536365,85123A,17850.0,6,12/1/2010 8:26,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
1,536365,71053,17850.0,6,12/1/2010 8:26,WHITE METAL LANTERN,3.39,United Kingdom,20.34
2,536365,84406B,17850.0,8,12/1/2010 8:26,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
3,536365,84029G,17850.0,6,12/1/2010 8:26,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
4,536365,84029E,17850.0,6,12/1/2010 8:26,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34


In [71]:
query1 = """
SELECT
    CustomerID,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY CustomerID
ORDER BY Total_Revenue DESC
LIMIT 5;
"""

top_customers_sql = pd.read_sql_query(
    query1,
    conn
)

display(top_customers_sql)

,CustomerID,Total_Revenue
0,18102.0,408759.96
1,14646.0,357531.09
2,17450.0,186197.03
3,14911.0,182797.24
4,16446.0,168472.50


In [72]:
query2 = """
SELECT
    Country,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY Country
ORDER BY Total_Revenue DESC;
"""

country_revenue_sql = pd.read_sql_query(
    query2,
    conn
)

display(country_revenue_sql)

,Country,Total_Revenue
0,United Kingdom,8.889920e+06
1,Netherlands,3.638845e+05
2,EIRE,3.319564e+05
3,Germany,2.640707e+05
4,France,2.270533e+05
5,Australia,1.739981e+05
6,Spain,6.744304e+04
7,Switzerland,6.661997e+04
8,Japan,4.860022e+04
9,Belgium,4.785802e+04


In [73]:
# Highest revenue generating product
product_revenue = (
    final_data
    .groupby("Description")["Revenue"]
    .sum()
)

best_product = product_revenue.idxmax()
best_product_revenue = product_revenue.max()


# Highest performing country
best_country = country_revenue.idxmax()
best_country_revenue = country_revenue.max()


# Customer with highest purchase value
customer_total_revenue = (
    final_data
    .groupby("CustomerID")["Revenue"]
    .sum()
)

best_customer = customer_total_revenue.idxmax()
best_customer_revenue = customer_total_revenue.max()


print("BUSINESS INSIGHTS\n")

print(
    f"1. The highest revenue generating product is "
    f"'{best_product}' with total revenue of "
    f"{best_product_revenue:.2f}."
)

print(
    f"\n2. '{best_country}' is the highest-performing "
    f"market with total revenue of "
    f"{best_country_revenue:.2f}."
)

print(
    f"\n3. Customer {best_customer} is the most valuable "
    f"customer with a total purchase value of "
    f"{best_customer_revenue:.2f}."
)

BUSINESS INSIGHTS

1. The highest revenue generating product is 'PAPER CRAFT , LITTLE BIRDIE' with total revenue of 168469.60.

2. 'United Kingdom' is the highest-performing market with total revenue of 8889920.30.

3. Customer 18102.0 is the most valuable customer with a total purchase value of 408759.96.


In [74]:
customer_category_summary = (
    customer_revenue
    .groupby("Customer_Category")
    .agg(
        Number_of_Customers=("CustomerID", "count"),
        Total_Revenue=("Revenue", "sum")
    )
    .sort_values(
        by="Total_Revenue",
        ascending=False
    )
)

display(customer_category_summary)

,Number_of_Customers,Total_Revenue
Customer_Category,,
Premium,53,3728618.971
Medium Value,1546,3439600.761
High Value,297,2527914.590
Low Value,2443,1085694.472


In [75]:
final_data.to_csv(
    "final_retail_sales.csv",
    index=False
)

print("Final dataset saved successfully!")

Final dataset saved successfully!


In [76]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

display(tables)

,name
0,retail_sales


In [77]:
conn.close()

print("Database connection closed successfully!")

Database connection closed successfully!


In [78]:
from google.colab import files

In [79]:
files.download("retail_sales.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [80]:
files.download("final_retail_sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [81]:
files.download("transactions.csv")
files.download("products.json")
files.download("customers.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>